## Windkessel-Informed Neural Network for Cuffless BP Estimation 
==========================================================================

Physics constraint: 2-element Windkessel diastolic decay equation
 ##   DBP = SBP * exp(-decay_time / RC) ##
 
This constrains DBP to be consistent with the model's own SBP prediction
and the measured diastolic decay time (systolic peak -> dicrotic notch).
SBP remains purely data-driven (see explanation in accompanying chat message).
 
Sections:

    -  Peak / dicrotic notch detection  (feature extraction from raw PPG)

    -  RC estimation                    (fit once from training data)

    -  Windkessel physics loss          (the actual PINN constraint)

    - Model architecture               (simple 1D CNN)

    - Training loop                    (combines data loss + physics loss)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import find_peaks

## Peak / Dicrotic Notch detection ##

Systolic peak = start of the "pressure is falling" phase, and dicrotic notch = the precise physiological start of diastole, where the heart is no longer pushing anything in.

For this eqn - P(t) = P_sys · exp(-t / RC)

We need t = decay time, which is simply the number of seconds it takes for arterial pressure to go from its systolic peak down to the start of the diastolic (passive) phase, measured as the time gap between the systolic peak and the dicrotic notch on the PPG waveform.

In [ ]:
def get_systolic_peak(ppg_window, fs=125):
    """
    Finds the systolic peak (main upstroke peak) in a single PPG window.
    Returns index of the peak, or None if not found.
    """
    peaks, props = find_peaks(
        ppg_window,
        height=np.mean(ppg_window),   # avoid noise floor
        distance=int(0.3 * fs)        # min 0.3s between peaks (~200 bpm cap)
    )
    if len(peaks) == 0:
        return None
    # take the highest peak as the systolic peak
    sys_idx = peaks[np.argmax(ppg_window[peaks])]
    return sys_idx

In [ ]:
def get_dicrotic_notch(ppg_window, sys_idx, fs=125):
    """
    Finds the dicrotic notch: the local minimum shortly after the
    systolic peak, before the diastolic (secondary) bump.
    Returns index of the notch, or None if not found.
    """
    if sys_idx is None:
        return None
 
    search_start = sys_idx + int(0.05 * fs)   # skip a few ms right after peak
    search_end = min(sys_idx + int(0.5 * fs), len(ppg_window) - 1)
    if search_start >= search_end:
        return None
 
    segment = ppg_window[search_start:search_end]
    # invert segment to find a "peak" = local minimum in original signal
    troughs, _ = find_peaks(-segment)
    if len(troughs) == 0:
        return None
 
    notch_idx = search_start + troughs[0]   # first local min after peak
    return notch_idx

In [ ]:
def get_decay_time(ppg_window, fs=125):
    """
    Returns diastolic decay time (systolic peak -> dicrotic notch), in seconds.
    Returns None if either landmark could not be detected (caller should
    skip / fall back to a dataset-median value for that sample).
    """
    sys_idx = get_systolic_peak(ppg_window, fs)
    notch_idx = get_dicrotic_notch(ppg_window, sys_idx, fs)
    if sys_idx is None or notch_idx is None:
        return None
    return (notch_idx - sys_idx) / fs

In [ ]:
def extract_decay_times(ppg_windows, fs=125, fallback='median'):
    """
    Runs decay-time extraction over a full array of PPG windows (N, L).
    Failed detections are filled with the median of successful ones.
    Returns: np.array of shape (N,)
    """
    decay_times = []
    for w in ppg_windows:
        dt = get_decay_time(w, fs)
        decay_times.append(dt)
 
    decay_times = np.array([d if d is not None else np.nan for d in decay_times])
    valid = decay_times[~np.isnan(decay_times)]
    if fallback == 'median' and len(valid) > 0:
        med = np.median(valid)
        decay_times = np.where(np.isnan(decay_times), med, decay_times)
    return decay_times.astype(np.float32)